# [Gripper] Dynamixel XM430-W210

## Import Libarary

In [1]:
import rby1_sdk as rby
from dynamixel_sdk import PortHandler, PacketHandler, COMM_SUCCESS
import time
from typing import Optional, List
import threading
import numpy as np

## Parameter settings

In [2]:
PROTOCOL_VERSION = 2.0
BAUDRATE = 2000000
DEVICE_NAME_GRIPPER = "/dev/rby1_gripper"

ADDR_OPERATING_MODE     = 11                # 0: Current Control / 1: Velocity Control / 3: Position Control(default)
ADDR_TORQUE_ENABLE      = 64                # 0: Turn OFF(default) / 1: Turn ON
ADDR_LED                = 65                # 0: Turn OFF(default) / 1: Turn ON
ADDR_GOAL_CURRENT       = 102
ADDR_GOAL_POSITION      = 116   
ADDR_PRESENT_CURRENT    = 126               # Unit: 2.69mA
ADDR_PRESENT_POSITION   = 132               # Unit: 1-pulse(or Tick)

CURRENT_CONTROL_MODE    = 0
POSTION_CONTROL_MODE    = 3
TORQUE_DISABLE          = 0
TORQUE_ENABLE           = 1

DXL_MIN_POS_VALUE       = 0
DXL_MAX_POS_VALUE       = 4095

RESOLUTION_TICKS_PER_REV    = 4096.0
TWO_PI                      = 2.0 * np.pi
RAD2DEG                     = 180.0 / np.pi

R_DXL_ID                    = 0
L_DXL_ID                    = 1

is_init_finish              = False

## Power ON the Robot

In [3]:
# ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot

robot = rby.create_robot_a(ROBOT_ADDRESS)   # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connect to the robot.")
    if (robot.power_on(".*") == True):         # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot.")
            
            # 여기 추가: 그리퍼 쪽 12V 출력 ON
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper.")
        else:
            print("Failed to power ON the Servo")

SUCCESS: Connect to the robot.
SUCCESS: Power ON the Robot.
SUCCESS: Power ON the Gripper.


## Power OFF the Robot

In [129]:
robot.set_tool_flange_output_voltage("right", 0)
robot.set_tool_flange_output_voltage("left", 0)
time.sleep(1)
robot.disable_control_manager()
robot.power_off(".*")

True

## Open the Gripper Port

In [4]:
port_hdlr = PortHandler(DEVICE_NAME_GRIPPER)
packet_hdlr = PacketHandler(PROTOCOL_VERSION)

if port_hdlr.openPort():
    print(f"SUCCESS: open the port: {DEVICE_NAME_GRIPPER}")
else:
    raise RuntimeError(f"Failed to open the port: {DEVICE_NAME_GRIPPER}")

if port_hdlr.setBaudRate(BAUDRATE):
    print(f"SUCCESS: change the baudrate {BAUDRATE}")
else:
    raise RuntimeError(f"Failed to change the baudrate")

SUCCESS: open the port: /dev/rby1_gripper
SUCCESS: change the baudrate 2000000


## Ping Test

In [5]:
dxl_activated = []

for dxl_id in [R_DXL_ID, L_DXL_ID]:
    dxl_model_num, dxl_comm_res, dxl_err = packet_hdlr.ping(port_hdlr, dxl_id)
    
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
    elif dxl_err != 0:
        print("[DXL ID:%d] %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
    else:
        dxl_activated.append(dxl_id)
        print("[DXL ID:%d] ping Succeeded. Dynamixel model number : %d" % (dxl_id, dxl_model_num))

[DXL ID:0] ping Succeeded. Dynamixel model number : 1030
[DXL ID:1] ping Succeeded. Dynamixel model number : 1030


### Continuous Ping Test

In [ ]:
success_cnt = 0
err_cnt = 0

for i in range(500):
    for dxl_id in [R_DXL_ID, L_DXL_ID]:
        dxl_model_num, dxl_comm_res, dxl_err = packet_hdlr.ping(port_hdlr, dxl_id)
        
        if dxl_comm_res != COMM_SUCCESS:
            err_cnt += 1
        elif dxl_err != 0:
            err_cnt += 1
        else:
            success_cnt += 1
        
print(f"Success: {success_cnt}, Error: {err_cnt}")


success cnt: 998, error cnt: 2


# Read & Write Register Functions

In [14]:
def DXL_write_torque_mode(dxl_id: int, torque_mode: int) -> None:
    dxl_comm_res, dxl_err = packet_hdlr.write1ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_TORQUE_ENABLE,
        int(torque_mode)
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
    else:
        if (torque_mode == TORQUE_DISABLE):      print("SUCCESS: Torque is disabled")
        elif (torque_mode == TORQUE_ENABLE):     print("SUCCESS: Torque is enabled")
        
        
def DXL_read_torque_mode(dxl_id: int) -> Optional[int]:
    val, dxl_comm_res, dxl_err = packet_hdlr.read1ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_TORQUE_ENABLE,
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
        return None
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
        return None
    else:
        torque_mode = val
        print(f"[DXL ID:{dxl_id}] SUCCESS: torque mode = {torque_mode}")
        return torque_mode    # 0: Disabled, 1: Enabled  


def DXL_write_operation_mode(dxl_id: int, operation_mode: int) -> None:
    dxl_comm_res, dxl_err = packet_hdlr.write1ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_OPERATING_MODE,
        int(operation_mode),
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
    else:
        if (operation_mode == CURRENT_CONTROL_MODE):      print("SUCCESS: Set Current Control Mode")
        elif (operation_mode == POSITION_CONTROL_MODE):   print("SUCCESS: Set Postion Control Mode")
    
    
def DXL_read_operation_mode(dxl_id: int) -> Optional[int]:
    val, dxl_comm_res, dxl_err = packet_handler.read1ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_OPERATING_MODE,
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
        return None
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
        return None
    else:
        operation_mode = val
        print(f"[DXL ID:{dxl_id}] SUCCESS: operation mode = {operation_mode}")
        return operation_mode    # 0: Current Control, 3: Position Control    

    

def DXL_read_position_rad(dxl_id: int) -> Optional[float]:
    val, dxl_comm_res, dxl_err = packet_hdlr.read4ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_PRESENT_POSITION,
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
        return None
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
        return None
    else:
        position_rad = float(position) / RESOLUTION_TICKS_PER_REV * TWO_PI 
        print(f"[DXL ID:{dxl_id}] SUCCESS: position = {position_rad}rad")
        return position_rad    # [rad]


# 1-unit = 2.69mA
def DXL_write_current_ampere(dxl_id: int, current_ampere: float) -> None:        
    current_value = int(current_ampere / 2.69 * 1000.0)
    dxl_comm_res, dxl_err = packet_hdlr.write2ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_GOAL_CURRENT,
        current_value,
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
    else:
        print(f"[DXL ID:{dxl_id}] SUCCESS: Set Current [{current_ampere} A]")
        

# 1-unit = 2.69mA
def DXL_read_current_ampere(dxl_id: int) -> None:        
    val, dxl_comm_res, dxl_err = packet_hdlr.read2ByteTxRx(
        port_hdlr,
        dxl_id,
        ADDR_PRESENT_CURRENT
    )
    if dxl_comm_res != COMM_SUCCESS:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getTxRxResult(dxl_comm_res)))
        return None
    elif dxl_err != 0:
        print("[DXL ID:%d] Fail %s" % (dxl_id, packet_hdlr.getRxPacketError(dxl_err)))
        return None
    else:
        current_A = float(val) * 2.69 / 1000.0 
        print(f"[DXL ID:{dxl_id}] SUCCESS: current = {current_A}A")
        return current_A     # [A]

# Enable Torque Mode

In [15]:
for dxl_id in dxl_activated:
    DXL_write_torque_mode(dxl_id, TORQUE_DISABLE)         # Operation mode 변경 전 torque OFF
    DXL_write_operation_mode(dxl_id, CURRENT_CONTROL_MODE)
    DXL_write_torque_mode(dxl_id, TORQUE_ENABLE)

SUCCESS: Torque is disabled
SUCCESS: Set Current Control Mode
SUCCESS: Torque is enabled
SUCCESS: Torque is disabled
SUCCESS: Set Current Control Mode
SUCCESS: Torque is enabled


In [ ]:
for dxl_id in dxl_activated:
    mode = read_operation_mode(dxl_id)
    torque = read_torque_enable(dxl_id)

In [ ]:

dxl_comm_res, dxl_err = packet_hdlr.write1ByteTxRx(port_hdlr, R_DXL_ID, ADDR_TORQUE_ENABLE, TORQUE_ENABLE)
if dxl_comm_res != COMM_SUCCESS:
    print("%s" % packet_hdlr.getTxRxResult(dxl_comm_res))
elif dxl_error != 0:
    print("%s" % packet_hdlr.getRxPacketError(dxl_err))
else:
    print("Dynamixel has been Successfully connected")


dxl_comm_res, dxl_err = packet_hdlr.read1ByteTxRx(port_hdlr, R_DXL_ID, ADDR_TORQUE_ENABLE)
if dxl_comm_res != COMM_SUCCESS:
    print("%s" % packet_hdlr.getTxRxResult(dxl_comm_res))
    print()
elif dxl_error != 0:
    print("%s" % packet_hdlr.getRxPacketError(dxl_err))
else:
    print("Dynamixel has been Successfully connected")
